In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from PIL import ExifTags, Image
from pycocotools.coco import COCO

In [ ]:
TACO_DIR = Path("/Users/hariomnarang/Desktop/personal/TACO/data")
ANN_FILE = TACO_DIR / "annotations.json"
EXP_BASE = Path("../../datasets/T007-uncentered/")
OUT_DIR = EXP_BASE / "data"
BIN_OUT = OUT_DIR / "binary"
OUT_DIR.mkdir(parents=True, exist_ok=True)
TEST_IMG_ID = 342
img_id = TEST_IMG_ID
coco = COCO(ANN_FILE)

# Scratch Coding

In [ ]:
import numpy as np
import cv2


def anns_to_mask(anns_sel, height, width, value=1):
    mask = np.zeros((height, width), dtype=np.uint8)
    for ann in anns_sel:
        for seg in ann["segmentation"]:
            poly = np.array(seg, dtype=np.int32).reshape(-1, 2)
            cv2.fillPoly(mask, [poly], value)
    return mask


def get_orientation_tag():
    for orientation in ExifTags.TAGS.keys():
        if ExifTags.TAGS[orientation] == "Orientation":
            return orientation
    return None


def load_image(image_path):
    # Obtain Exif orientation tag code
    orientation = get_orientation_tag()

    img = Image.open(image_path)

    # Load and process image metadata
    if img._getexif() and orientation:
        exif = dict(img._getexif().items())
        # Rotate portrait and upside down images if necessary
        if orientation in exif:
            if exif[orientation] == 3:
                img = img.rotate(180, expand=True)
            if exif[orientation] == 6:
                img = img.rotate(270, expand=True)
            if exif[orientation] == 8:
                img = img.rotate(90, expand=True)

    img = img.convert("RGB")
    return np.array(img)


def extract_mask_for_image_id(img_id, coco, taco_dir):
    image_path = taco_dir / coco.loadImgs(img_id)[0]["file_name"]
    annIds = coco.getAnnIds(imgIds=img_id, catIds=[], iscrowd=None)
    anns_sel = coco.loadAnns(annIds)
    img_array = load_image(image_path)
    h, w = img_array.shape[:2]
    mask = anns_to_mask(anns_sel, h, w)
    return img_array, mask

In [ ]:
from mtrain.smallnet.unet.predict import overlay_mask_on_img

img_arr, mask = extract_mask_for_image_id(342, coco, TACO_DIR)
img_arr = overlay_mask_on_img(img_arr, mask.astype(bool))

plt.imshow(img_arr)

In [ ]:
pilimg = Image.fromarray(img_arr, "RGB")
pilmask = Image.fromarray(mask, "L")

In [ ]:
pilimg.save("./img.jpeg")
pilmask.save("./mask.png")

In [ ]:
pilmask = Image.open("./mask.png")
if pilmask.mode != "L":
    raise

In [ ]:
plt.imshow(np.array(pilmask))

In [ ]:
img_arr[mask.astype(bool)] = [255, 0, 0]
plt.imshow(img_arr)

In [ ]:
image_path = TACO_DIR / coco.loadImgs(img_id)[0]["file_name"]

In [ ]:
img_array = load_image(image_path)
plt.imshow(img_array)

In [ ]:
img_array.shape

In [ ]:
annIds = coco.getAnnIds(imgIds=img_id, catIds=[], iscrowd=None)
anns_sel = coco.loadAnns(annIds)
h, w = img_array.shape[:2]
mask = anns_to_mask(anns_sel, h, w)

In [ ]:
plt.imshow(img_array)

In [ ]:
plt.imshow(mask)

In [ ]:
from mtrain.smallnet.unet.extract.taco_to_fastai import extract_taco_dataset
from mtrain.smallnet.unet.extract.draw import show_extracted_dataset


In [ ]:
extract_taco_dataset(
    ann_file=ANN_FILE,
    taco_dir=TACO_DIR,
    out_dir=OUT_DIR / "binary",
    should_collapse_mask_to_binary=True,
)

# BBox based rsz, experiment

In [ ]:
def get_bounding_boxes_connected(binary_mask):
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary_mask, connectivity=8
    )

    boxes = []
    for label in range(1, num_labels):  # skip label 0 (background)
        x = stats[label, cv2.CC_STAT_LEFT]
        y = stats[label, cv2.CC_STAT_TOP]
        w = stats[label, cv2.CC_STAT_WIDTH]
        h = stats[label, cv2.CC_STAT_HEIGHT]
        boxes.append((label, x, y, w, h))

    return boxes

In [ ]:
boxes = get_bounding_boxes_connected(mask)

In [ ]:
def show(crops, figsize=None, ncols=2):
    crops = list(crops)
    rows = math.ceil(len(crops) / ncols)
    if figsize is None:
        figsize = (10 * rows, 10 * rows)
        print("figsize", figsize)
    _, axs = plt.subplots(rows, ncols, figsize=figsize)
    axs = axs.flatten()
    for i, c in enumerate(crops):
        axs[i].imshow(c)
    plt.tight_layout()
    plt.show()

In [ ]:
import random
from dataclasses import dataclass
import math


@dataclass
class Bbox:
    x: int
    y: int
    w: int
    h: int

    @property
    def x2(self):
        return self.x + self.w

    @property
    def y2(self):
        return self.y + self.h


# given a bbox, we want to resize it to some size.
# i have the final shape of the image i want (that would be my cell size)
# my original algorithm has 2 bboxes.
# it first decides the alpha based on their heights
# it then finds how much space the top, bottom, left, right take in the target
# it scales them to our size using the alpha
# then adds them to our bbox. This gives us a crop which can be resized to the targets actual size
# we want without target bbox
# we essentially have the height we want for the object
# we have the final cell size
# using bbox height and required height, we get the aspect ratio
# cell height - our calc height is leftover
# cell width - our calc width is leftover
# split leftover in 2 parts, randomly
# and then create a crop to resize
# easy


def get_engulfing_bbox_to_resize(
    mask: np.ndarray,
    bbox: Bbox,
    target_bbox_height: int,
    target_cell_height: int,
    target_cell_width: int,
):
    aspect_ratio = target_bbox_height / bbox.h
    print("AR", aspect_ratio)
    target_bbox_width = math.ceil(aspect_ratio * bbox.w)
    print("original height and width", bbox.h, bbox.w)
    print("target_height and width", target_bbox_height, target_bbox_width)

    if target_bbox_height > target_cell_height:
        raise Exception(
            f"target_bbox_height > target_cell_height: {target_bbox_height} > {target_cell_height}"
        )
    if target_bbox_width > target_cell_width:
        raise Exception(
            f"target_bbox_width > target_cell_width: {target_bbox_width} > {target_cell_width}"
        )

    leftover_height = target_cell_height - target_bbox_height
    leftover_height = math.ceil(leftover_height / aspect_ratio)

    leftover_width = target_cell_width - target_bbox_width
    leftover_width = math.ceil(leftover_width / aspect_ratio)

    top, bottom = _split_dist_in_2(leftover_height)
    left, right = _split_dist_in_2(leftover_width)

    x1 = max(bbox.x - left, 0)
    x2 = min(bbox.x2 + right, mask.shape[1])
    y1 = max(bbox.y - top, 0)
    y2 = min(bbox.y2 + bottom, mask.shape[0])

    return Bbox(x=x1, y=y1, w=x2 - x1, h=y2 - y1)


def _split_dist_in_2(dist: int) -> tuple[int, int]:
    splitter = random.randint(0, dist)
    return splitter, dist - splitter


def resize_bbox_in_img(
    img, bbox: Bbox, target_height, target_width, interp=cv2.INTER_AREA
):
    crop = img[bbox.y : bbox.y2, bbox.x : bbox.x2]
    return cv2.resize(crop, (target_height, target_width), interpolation=interp)


In [ ]:
lbl, x, y, w, h = boxes[0]
bbox = Bbox(x, y, w, h)

bbox_to_resize = get_engulfing_bbox_to_resize(mask, bbox, 10, 100, 100)
print(bbox_to_resize)
rsz_img = resize_bbox_in_img(cv2.GaussianBlur(img_array, (5,5), 0), bbox_to_resize, 100, 100)
plt.imshow(rsz_img)

In [ ]:
def get_batch_from_extracted_dataset(d, n=8, img_id=None):
    ims, msks = d / "images", d / "masks"
    res = []
    it = ims.glob("*")
    ims = list(it)
    random.shuffle(ims)
    for im in ims[:n]:
        msk = msks / f"{im.stem}.png"
        res.append((im, msk))
    return res

In [ ]:
batch =  get_batch_from_extracted_dataset(OUT_DIR / "binary")

In [ ]:
idx = 3
img, mask = plt.imread(batch[idx][0]), np.array(Image.open(batch[idx][1]).convert("L"))

show([img, mask])

In [ ]:
TARGET_H = 50
TARGET_BBOX_H = 10
bboxes = get_bounding_boxes_connected(mask)
res = []
for bbox in bboxes:
    lbl, x, y, w, h = bbox
    bb = get_engulfing_bbox_to_resize(mask, Bbox(x,y,w,h), TARGET_BBOX_H, TARGET_H, TARGET_H)
    res.append(resize_bbox_in_img(cv2.GaussianBlur(img, (5,5), 0), bb, TARGET_H, TARGET_H))
    
show(res, ncols=3)

In [ ]:
show_extracted_dataset(OUT_DIR / "binary")

# DLS

In [ ]:
from mtrain.smallnet.unet.train import get_dls

In [ ]:
dls = get_dls(
    8,
    EXP_BASE / "test-log",
    100,
    OUT_DIR / "binary" / "images",
    OUT_DIR / "binary" / "masks",
)

In [ ]:
dls.show_batch()

# Cropping Manual (code here)

In [ ]:
import math


def extract_single_crop(coco, img_path, mask_path, max_padding, horiz_skew, vert_skew):
    # horiz_skew: relation between the left and right padding max values
    # if horiz_skew is negative, then we increase left padding with the skew scale compared to right (fix left max to max_padding, chjange right max to left_max / skew)
    # same for positive skew

    # Pick a random annotation to center the crop around
    img_id = int(img_path.stem)
    img = Image.open(img_path)
    img_array = np.array(img)
    img_height, img_width = img_array.shape[:2]

    mask = Image.open(mask_path)
    mask_array = np.array(mask)

    ann_ids = coco.getAnnIds(imgIds=img_id)
    anns = coco.loadAnns(ann_ids)

    ann = random.choice(anns)
    x, y, w, h = ann["bbox"]

    # Random padding on each side
    print("skews", horiz_skew, vert_skew)
    max_left, max_right = _get_paddings(max_padding, horiz_skew)
    max_top, max_bottom = _get_paddings(max_padding, vert_skew)
    print(max_left, max_right, max_top, max_bottom)
    pad_left = random.randint(0, max_left)
    pad_right = random.randint(0, max_right)
    pad_top = random.randint(0, max_top)
    pad_bottom = random.randint(0, max_bottom)

    # Calculate crop boundaries
    crop_x1 = max(0, int(x - pad_left))
    crop_y1 = max(0, int(y - pad_top))
    crop_x2 = min(img_width, int(x + w + pad_right))
    crop_y2 = min(img_height, int(y + h + pad_bottom))

    # Crop image and mask
    crop_img = img_array[crop_y1:crop_y2, crop_x1:crop_x2]
    crop_mask = mask_array[crop_y1:crop_y2, crop_x1:crop_x2]
    return crop_img, crop_mask
    # resizer = PaddedResize(crop_size)
    # crop_img_resized = resizer(crop_img)
    # crop_mask_resized = resizer(crop_mask)

    # crop_bbox = [crop_x1, crop_y1, crop_x2 - crop_x1, crop_y2 - crop_y1]
    # return (crop_img_resized, crop_mask_resized, crop_bbox)


def _get_paddings(max_padding, skew):
    # we return before_padding and after_padding
    before = max_padding
    if skew < 0:
        before = max_padding
        after = math.floor(before / (-skew))
    else:
        after = max_padding
        before = math.floor(after / skew)
    return before, after


In [ ]:
BIN_OUT = OUT_DIR / "binary"
coco = COCO(ANN_FILE)
image = BIN_OUT / "images" / "10.jpeg"
mask = BIN_OUT / "masks" / "10.png"


In [ ]:
crops = []
for i in range(8):
    horiz_skew = random.choice([-1, 1]) * random.uniform(1, 3)
    vert_skew = random.choice([-1, 1]) * random.uniform(1, 3)
    crops.append(
        extract_single_crop(
            coco, image, BIN_OUT / "masks" / "10.png", 1000, horiz_skew, vert_skew
        )
    )


_, ax = plt.subplots(8, 2)
for i in range(8):
    ax[i][0].imshow(crops[i][0])
    ax[i][1].imshow(crops[i][1])
plt.show()


In [ ]:
ci, cm = extract_single_crop(coco, image, BIN_OUT / "masks" / "10.png", 1000)
_, ax = plt.subplots(2, 2)
ax[0][0].imshow(plt.imread(image))
ax[0][1].imshow(plt.imread(mask))
ax[1][0].imshow(ci)
ax[1][1].imshow(cm)

In [ ]:
plt.imshow(ci)

# Cropping: lib

In [ ]:
from mtrain.smallnet.unet.extract.cropping import (
    engulf,
    cut,
    extract_crops_for_single_image,
)
from pathlib import Path

In [ ]:
BIN_OUT = Path("../../datasets/T007-uncentered/data/EXT")
BIN_OUT, TEST_IMG_ID

## Test engulf

In [ ]:
from mtrain.smallnet.unet.extract.cropping import extract_crops_for_single_image

TEST_IMG_PATH = BIN_OUT / Path("images/342.jpeg")
TEST_MASK_PATH = BIN_OUT / "masks" / f"{TEST_IMG_PATH.stem}.png"

print(TEST_IMG_PATH.exists(), TEST_MASK_PATH.exists())

crops = extract_crops_for_single_image(
    coco, TEST_IMG_PATH, TEST_MASK_PATH, 3, 10, 3, "engulf"
)
_, ax = plt.subplots(len(crops), 2, figsize=(40, 40))
for i, (img, mask) in enumerate(crops):
    ax[i][0].imshow(img)
    ax[i][1].imshow(mask)
plt.tight_layout()
plt.show()

# crop_img, crop_mask = engulf.extract_single_crop(
#     coco, TEST_IMG_PATH, TEST_MASK_PATH, 3, 1, 1, min_padding=100, max_padding=100
# )

## Test cut


In [ ]:
TEST_IMG_PATH = BIN_OUT / "images" / "351.jpeg"
TEST_MASK_PATH = BIN_OUT / "masks" / "351.png"

In [ ]:
plt.imshow(plt.imread(TEST_IMG_PATH))

In [ ]:
crop_img, crop_mask = cut.extract_crop_by_cutting_object(
    coco, TEST_IMG_PATH, TEST_MASK_PATH, 3
)
_, ax = plt.subplots(1, 2)
ax[0].imshow(crop_img)
ax[1].imshow(crop_mask)

# Test full single image

In [ ]:
def resize_and_pad_raw(img, target_size, pad_value=0):
    """
    Resize image so that the larger dimension becomes `target_size`,
    then pad the other dimension to get
    (target_size x target_size).

    Args:
        img (np.ndarray): Input image
        target_size (int): Output size (e.g. 50)
        pad_value (int or tuple): Padding value (default = 0 / black)

    Returns:
        np.ndarray: (target_size x target_size) image
    """
    h, w = img.shape[:2]

    # Scale so max dimension == target_size
    scale = target_size / max(h, w)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))

    # blurred = cv2.GaussianBlur(img, ksize=(3, 3), sigmaX=3, sigmaY=3)
    # blurred = cv2.GaussianBlur(img, (0,0), 6)
    # blurred = cv2.GaussianBlur(blurred, (0,0), 11)
    # blurred = cv2.GaussianBlur(img, (0,0), 13)
    # blurred = cv2.GaussianBlur(blurred, (0,0), 2)
    def disk_kernel(r):
        size = 2 * r + 1
        k = np.zeros((size, size), np.float32)
        cv2.circle(k, (r, r), r, 1, -1)
        return k / k.sum()

    blurred = cv2.filter2D(img, -1, disk_kernel(4))
    resized = cv2.resize(blurred, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Compute padding
    pad_x = target_size - new_w
    pad_y = target_size - new_h

    pad_left = pad_x // 2
    pad_right = pad_x - pad_left
    pad_top = pad_y // 2
    pad_bottom = pad_y - pad_top

    padded = cv2.copyMakeBorder(
        resized,
        pad_top,
        pad_bottom,
        pad_left,
        pad_right,
        borderType=cv2.BORDER_CONSTANT,
        value=pad_value,
    )

    meta = {
        "orig_h": h,
        "orig_w": w,
        "scale": scale,
        "new_h": new_h,
        "new_w": new_w,
        "pad_left": pad_left,
        "pad_right": pad_right,
        "pad_top": pad_top,
        "pad_bottom": pad_bottom,
        "target_size": target_size,
    }

    return padded, meta

In [ ]:
plt.imshow(resize_and_pad_raw(im, 50)[0])

In [ ]:
IMS0 = "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/positive-samples/133855545702381.jpg"
im = plt.imread(IMS0)
plt.imshow(im)

In [ ]:
plt.imshow(resize_and_pad_raw(im[400:500, 100:200], 50)[0])

In [ ]:
plt.imshow(im[400:500, 100:200])

In [ ]:
plt.imshow(resize_and_pad_raw(img[600:, 750:1500], 50)[0])

In [ ]:
import functools
from mtrain.smallnet.unet.extract.cropping import engulf, v2_engulf

In [ ]:
img, mask = v2_engulf.extract_single_crop(
    coco, TEST_IMG_PATH, TEST_MASK_PATH, 10, 1, 3000
)

In [ ]:
plt.imshow(img[600:, 750:1500])

In [ ]:
plt.imshow(resize_and_pad_raw(img[600:, 750:1500], 50)[0])

In [ ]:
plt.imshow(resize_and_pad_raw(img, 50)[0])

In [ ]:
from mtrain.smallnet.unet.extract.cropping import extract_crops_for_single_image
from mtrain.smallnet.unet.extract.draw import show_extracted_dataset
from mtrain.smallnet.tfms import PaddedResize


def single_image_test(img_path, mask_path, crop_size):
    crops = extract_crops_for_single_image(
        coco, img_path, mask_path, 10, 10, mode="engulf"
    )
    resizer = PaddedResize(crop_size)

    res = []
    for img, mask in crops:
        try:
            img, mask = resizer(img), resizer(mask)
            res.append((img, mask))
        except Exception as ex:
            print("image shape", img.shape, "mask shape", mask.shape)
            print("reason", str(ex))
            raise

    return res

In [ ]:
FNAME = "352"
CROP_SIZE = 40
TEST_IMG_PATH = Path(
    f"/Users/hariomnarang/Desktop/personal/roads/datasets/T007-uncentered/data/EXT/images/{FNAME}.jpeg"
)
TEST_MASK_PATH = Path(
    f"/Users/hariomnarang/Desktop/personal/roads/datasets/T007-uncentered/data/EXT/masks/{FNAME}.png"
)

plt.imshow(plt.imread(TEST_IMG_PATH))

In [ ]:
res = single_image_test(TEST_IMG_PATH, TEST_MASK_PATH, CROP_SIZE)

In [ ]:
_, ax = plt.subplots(len(res), 1, figsize=(800, 200))
for i in range(len(res)):
    ax[i].imshow(res[i][0])
# plt.imshow(res[3][0])

In [ ]:
_, ax = plt.subplots(len(crops), 2, figsize=(30, 50))
for i in range(len(crops)):
    ax[i][0].imshow(crops[i][0])
    ax[i][1].imshow(crops[i][1])
plt.show()

# Test DS extraction

In [ ]:
from mtrain.smallnet.unet.extract.cropping import create_crops_dataset
from mtrain.smallnet.unet.extract.taco_to_fastai import extract_taco_dataset
from mtrain.smallnet.unet.extract.draw import show_extracted_dataset, show

In [ ]:
# orig_dir = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/T007-uncentered/engulfed-bbox-levels-crops")
# dest_dir = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/T007-uncentered/filtered-engulfed-bbox-levels-crops")

# from mtrain.smallnet.unet.extract.cropping.filters import filter_small_objects_in_ds
# filter_small_objects_in_ds(orig_dir, dest_dir, 85)

In [ ]:
import shutil

EXT_DIR = OUT_DIR / "EXT"
CROPS_DIR = OUT_DIR.parent / "128x128-all-data"
engulfed_dir = CROPS_DIR


In [ ]:
EXT_DIR

In [ ]:
shutil.rmtree(EXT_DIR, ignore_errors=True)
shutil.rmtree(CROPS_DIR, ignore_errors=True)

In [ ]:
extract_taco_dataset(
    ann_file=ANN_FILE,
    taco_dir=TACO_DIR,
    out_dir=EXT_DIR,
    should_collapse_mask_to_binary=True,
)

In [ ]:
show_extracted_dataset(EXT_DIR)

In [ ]:
crop_size = 128
# small_crops = list(range(5, 40, 5))

big_levels = list(range(5, 100, 5))


lvls = big_levels
lvls

In [ ]:
eimgs = list(Path("/Users/hariomnarang/Desktop/personal/roads/datasets/T007-uncentered/data/taco_trash_200/images").glob("*"))

In [ ]:
engulfed_dir

In [ ]:
if engulfed_dir.exists():
    shutil.rmtree(engulfed_dir)
print("creating at", engulfed_dir)
create_crops_dataset(
    ANN_FILE,
    EXT_DIR,
    engulfed_dir,
    workers=4,
    crop_size=224,
    bbox_heights=lvls,
    min_area=40,
)

In [ ]:
engulfed_dir

In [ ]:
# level filter
def level_in_range(lvl_low, lvl_high):
    def wrapped(img):
        lvl = int(Path(img).stem.split("_")[0])
        return lvl_low <= lvl <= lvl_high
    return wrapped

show_extracted_dataset(engulfed_dir, filterer=level_in_range(5,10))

In [ ]:
pat_imgs = list((engulfed_dir / "images").glob("ImhYqtcn_*"))
imgs_and_masks = [
    (pat_img, engulfed_dir / "masks" / f"{pat_img.stem}.png")
    for pat_img in pat_imgs
]
imgs_and_masks = sorted(imgs_and_masks)
res = []
for img, mask in imgs_and_masks[:10]:
    res.append(np.array(Image.open(img)))
    res.append(np.array(Image.open(mask).convert("L")))

show(
    res, ncols=2
)

In [ ]:
show_extracted_dataset(engulfed_dir)

In [ ]:
engulfed_dir

In [ ]:
dest = Path('../../datasets/T007-uncentered/engulfed-bbox-levels-crops')

In [ ]:
import shutil
from tqdm import tqdm
for img in tqdm(list((engulfed_dir / "masks").glob("*"))):
    shutil.copy(img, dest / "masks")

In [ ]:
show_extracted_dataset(dest)

In [ ]:
! dvc add '../../datasets/T007-uncentered/data/engulfed-minpad_3000-size_200/'

In [ ]:
! dvc add '../../datasets/T007-uncentered/only_engulfed'

# BBox testing

In [ ]:
from mtrain.smallnet.unet.extract.cropping import bbox_based

In [ ]:
! ls {EXT_DIR}/masks/0.png

In [ ]:
from mtrain.disk import DiskImage, DiskBooleanMask
from mtrain.utils import show, it_chain


img = DiskImage.load(EXT_DIR / "images" / "4.jpeg")
mask = DiskBooleanMask.load(EXT_DIR / "masks" / "4.png")

images_and_masks = list(bbox_based.extract_crops_for_single_image(img, mask, [10,20,30,50,80], 100, 100))

show([img, mask])

In [ ]:
show(it_chain(((i,m) for (i,m) in images_and_masks[:4] if i is not None)), (40,40), ncols=4)